Chapter 07. 실행되는 모델에서 검증된 모델로 — 한국어 텍스트 분류·추천 개선하기

### 실습 1. 개선 데이터 확인하기

**학습목표:**
모델을 변경하기 전, 카테고리별 균형 데이터의 구성과 형태(분야별 분포, 결측치, 중복 여부)를 파악합니다.

In [1]:
import pandas as pd

# 데이터 불러오기
from pathlib import Path

project_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "raw").exists())
data_path = project_root / "data" / "raw" / "교보문고_카테고리균형_도서데이터.csv"
df = pd.read_csv(data_path, encoding="utf-8-sig").rename(
    columns={"title": "상품명", "category": "분야"}
)
print("데이터 형태:", df.shape)
print("컬럼 목록:", df.columns)
display(df.head())

# 분야별 도서 수 확인
print("\n[분야별 분포]")
print(df["분야"].value_counts())

# 결측치와 중복 확인
print("\n[결측치 확인]")
print(df.isna().sum())
print("중복 상품명:", df["상품명"].duplicated().sum())

데이터 형태: (1692, 11)
컬럼 목록: Index(['rank', 'productCode', 'productId', '상품명', 'listPrice', 'salePrice',
       'author', 'publisher', 'publicationDate', 'publicationYear', '분야'],
      dtype='str')


,rank,productCode,productId,상품명,listPrice,salePrice,author,publisher,publicationDate,publicationYear,분야
0,1,9788915008601,S000220700678,임신 출산 육아 대백과(2026-2027),19500.0,17550.0,삼성출판사 편집부,삼성출판사,20260730,2026,가정/육아
1,2,9791122019827,S000220939598,세상을 움직이는 아이들은 무엇이 다른가,18800.0,16920.0,임지은,다빈치페이지,20260910,2026,가정/육아
2,3,9791164053704,S000221030617,멈추지 못하는 아이들,18500.0,16650.0,선자연,북하우스,20260907,2026,가정/육아
3,4,9791168127531,S000208779616,최민준의 아들코칭 백과,18800.0,16920.0,최민준,위즈덤하우스,20230830,2023,가정/육아
4,5,9791171174348,S000213691665,0~3세 기적의 뇌과학 육아,20000.0,18000.0,그리어 커센바움,21세기북스,20240703,2024,가정/육아



[분야별 분포]
분야
가정/육아        188
건강           188
경제/경영        188
소설           188
시/에세이        188
요리           188
인문           188
취미/실용/스포츠    188
컴퓨터/IT       188
Name: count, dtype: int64

[결측치 확인]
rank               0
productCode        0
productId          0
상품명                0
listPrice          0
salePrice          0
author             0
publisher          0
publicationDate    0
publicationYear    0
분야                 0
dtype: int64
중복 상품명: 10


**실습결과 요약:**
* 9개 분야가 각각 188권으로 고르게 분포되어 데이터 쏠림 현상이 없음을 확인했습니다.
* 모델 개선은 단순히 알고리즘 변경뿐만 아니라, 데이터, 텍스트 표현, 결과 처리 로직 등 전체적인 요소를 모두 점검하고 다루는 과정임을 인지합니다.

### 실습 2. Baseline을 먼저 측정하기

**학습목표:**
개선 효과를 객관적으로 비교하기 위한 기준점(Baseline)을 설정합니다. 형태소 분석 전 원본 제목 텍스트에 기본 TF-IDF와 Naive Bayes를 적용하여 초기 성능을 측정합니다.

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report

# 1. Train / Test 분리
X = df["상품명"]
y = df["분야"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)

# 2. 훈련 데이터 학습 (TF-IDF 변환)
baseline_vectorizer = TfidfVectorizer()
X_train_vec = baseline_vectorizer.fit_transform(X_train)
X_test_vec = baseline_vectorizer.transform(X_test)

# 3. 모델 학습 및 예측
baseline_model = MultinomialNB()
baseline_model.fit(X_train_vec, y_train)
baseline_pred = baseline_model.predict(X_test_vec)

# 4. 성능 평가
baseline_accuracy = accuracy_score(y_test, baseline_pred)
print("Baseline Accuracy:", baseline_accuracy)
print(classification_report(y_test, baseline_pred))

Baseline Accuracy: 0.49852507374631266
              precision    recall  f1-score   support

       가정/육아       0.59      0.63      0.61        38
          건강       0.21      0.54      0.30        37
       경제/경영       0.67      0.47      0.55        38
          소설       0.52      0.34      0.41        38
       시/에세이       0.47      0.18      0.26        38
          요리       0.67      0.54      0.60        37
          인문       0.47      0.37      0.41        38
   취미/실용/스포츠       0.80      0.54      0.65        37
      컴퓨터/IT       0.67      0.87      0.76        38

    accuracy                           0.50       339
   macro avg       0.56      0.50      0.51       339
weighted avg       0.56      0.50      0.51       339



**실습결과 요약:**
* Baseline 성능: Accuracy 49.85%, Macro F1 0.51, 정답 수 169 / 339권.
* 이 숫자가 이후 형태소 분석 등 전처리를 적용했을 때 모델이 얼마나 개선되었는지 판단하는 비교 기준점이 됩니다.

### 실습 3. 한국어 형태소 분석 적용하기

**학습목표:**
기본 문자열을 그대로 사용하지 않고, 한국어 제목에서 의미 있는 형태소(일반 명사, 고유 명사, 영문 등)만 추출하여 분류에 유리하도록 텍스트 표현을 변경해 봅니다.

In [3]:
# 형태소 분석기 설치가 필요한 경우 터미널에서 실행: python -m pip install kiwipiepy
from kiwipiepy import Kiwi

kiwi = Kiwi()
sample = "처음 배우는 파이썬 데이터 분석"

# 형태소 추출 테스트
for token in kiwi.tokenize(sample):
    print(token.form, token.tag)

# 실제 데이터(NNG 일반명사, NNP 고유명사, SL 영문 등 추출) 적용 결과 확인
def tokenize_title(text):
    return " ".join(token.form for token in kiwi.tokenize(str(text)) if token.tag in {"NNG", "NNP", "SL"})

df["상품명_토큰"] = df["상품명"].map(tokenize_title)
display(df[["상품명", "상품명_토큰"]].head(10))

처음 NNG
배우 VV
는 ETM
파이썬 NNP
데이터 NNG
분석 NNG


,상품명,상품명_토큰
0,임신 출산 육아 대백과(2026-2027),임신 출산 육아 백과
1,세상을 움직이는 아이들은 무엇이 다른가,세상 아이
2,멈추지 못하는 아이들,아이
3,최민준의 아들코칭 백과,최민준 아들 코칭 백과
4,0~3세 기적의 뇌과학 육아,기적 뇌 과학 육아
5,사춘기 아이와 대화하는 법은 따로 있습니다,사춘기 아이 대화 법
6,가장 완벽한 ADHD 양육 가이드,완벽 ADHD 양육 가이드
7,삐뽀삐뽀 119 소아과,소아과
8,불편함을 다룰 줄 아는 아이,불편 아이
9,잘 자고 잘 먹는 아기의 시간표,아기 시간표


**실습결과 요약:**
* 한국어 형태소 분석기(Kiwi)를 통해 문장을 의미 단위로 분리했습니다.
* 중요한 것은 형태소 분석기의 단순 사용이 아니라, TF-IDF에 들어가는 텍스트 표현을 바꾸었을 때 실제 분류 성능 평가 결과가 달라지는지 확인하는 것입니다.